# Build Your First AI Agent

In this notebook we build a small **tool-using agent from scratch** — no agent
framework — around a **local Qwen3-VL** model (`Qwen/Qwen3-VL-8B-Instruct`, the
same model family used in the assignment). You will see every part of the
agentic loop explicitly: the system prompt, the tool registry, parsing the
model's tool calls, running the tool, and feeding the observation back.

The conceptual loop (Thought → Action → Observation → … → Final) is illustrated below.


![Image](https://d2908q01vomqb2.cloudfront.net/ca3512f4dfa95a03169c5a670a4c91a19b3077b4/2025/05/16/agentic-loop.png)

## Install libraries

We run the model **locally** with 🤗 `transformers` — no cloud SDK or credentials.
Qwen3-VL requires a recent `transformers` release.

In [ ]:
# Pick a GPU BEFORE importing torch (change "0" to a free GPU index; check with nvidia-smi)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

%pip install -q -U "transformers>=4.57.0" accelerate bitsandbytes pillow

## Load the model (local Qwen3-VL)

The first run downloads the weights (~16 GB) into `~/.cache/huggingface`. We load
in 4-bit so it fits comfortably on a single ~18 GB GPU. This is the same model
family and loading pattern used in the assignment notebook.

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODEL_ID = os.getenv("QWEN_MODEL_ID", "Qwen/Qwen3-VL-8B-Instruct")
print("Loading", MODEL_ID, "on", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Loaded. VRAM (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

In [ ]:
from typing import Any


def qwen_chat(system_prompt: str, user_prompt: str, *, history: list[dict[str, Any]] | None = None,
              temperature: float = 0.0, max_tokens: int = 800) -> str:
    """Single call to the local Qwen model. Returns the assistant's text.

    `history` is an optional list of prior turns in Qwen chat format
    ([{"role": ..., "content": [{"type": "text", "text": ...}]}]).
    """
    chat = [{"role": "system", "content": [{"type": "text", "text": system_prompt}]}]
    if history:
        chat.extend(history)
    if user_prompt:
        chat.append({"role": "user", "content": [{"type": "text", "text": user_prompt}]})

    text = processor.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], return_tensors="pt").to(model.device)
    gen_kwargs = {"max_new_tokens": max_tokens}
    if temperature and temperature > 0:
        gen_kwargs.update(do_sample=True, temperature=temperature)
    else:
        gen_kwargs.update(do_sample=False)
    with torch.no_grad():
        generated = model.generate(**inputs, **gen_kwargs)
    trimmed = generated[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

## Quick check (model responds)

In [ ]:
print(qwen_chat("You are a helpful assistant. Answer in one short sentence.",
                "In one sentence, what is an AI agent?"))

## Part 1 - A plain LLM (no tools)

First, let's see what the model does **without any tools** — just a system prompt.
Notice it can explain concepts, but it *cannot* know the current time and may
miscount letters, because it has no way to run code or look things up.

In [ ]:
# A specialized system prompt (no tools yet)
SUBJECT_EXPERT_SYSTEM = """You are a Computer Science Subject Expert specializing
in explaining technical concepts clearly and concisely. Your expertise
covers programming languages, data structures, algorithms, computer
architecture, and software engineering principles.

When explaining concepts:
1. Start with a clear, concise definition
2. Provide short, but relevant examples to illustrate the concept
3. Explain practical applications where applicable
4. Avoid unnecessary jargon, but introduce important terminology
5. Consider the learner's perspective and make complex topics accessible
"""

In [ ]:
query = """
Answer the following questions:
1. What is the current time in UTC?
2. Which CS concept can be traced back to Paul Bachmann?
3. Tell me how many letter R's are in the word "strawberry"
"""

print(qwen_chat(SUBJECT_EXPERT_SYSTEM, query))

In [ ]:
print("Observe: the model above cannot know the real current time, and it may "
      "miscount letters. Next we give it TOOLS and a loop so it can act.")

## Part 2 - Give the model tools + a loop

An **agent** = an LLM + tools + a loop that lets it *act*. We'll:

1. Define a few Python **tools** and register them in a dict.
2. Tell the model, in the system prompt, how to request a tool (a simple
   `Action` / `Action Input` text protocol — the same one used in the
   assignment and the reflexion notebook).
3. Write the loop: call the model → parse a tool call → run the tool → feed the
   observation back → repeat until the model emits `Final:`.

In [ ]:
import ast, operator, datetime

# --- Tool 1: a safe calculator (no eval) ---
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
        ast.Mod: operator.mod, ast.Pow: operator.pow, ast.USub: operator.neg}

def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp):
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("unsupported expression")

def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression, e.g. "3111696 / 74088"."""
    return str(_safe_eval(ast.parse(str(expression), mode="eval")))

# --- Tool 2: current UTC time ---
def current_time() -> str:
    """Return the current UTC time in ISO 8601 format."""
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

# --- Tool 3: count a letter in a word ---
def letter_counter(word: str, letter: str) -> str:
    """Count occurrences of `letter` in `word` (case-insensitive)."""
    return str(word.lower().count(letter.lower()))

# Tool registry: name -> function
TOOLS = {
    "calculator": calculator,
    "current_time": current_time,
    "letter_counter": letter_counter,
}
print("Agent tools:", sorted(TOOLS))

In [ ]:
import json, re

ACTION_RE = re.compile(r"^Action:\s*(?P<name>\w+)\s*$", re.MULTILINE)
FINAL_RE = re.compile(r"^Final:\s*(?P<final>[\s\S]+)$", re.MULTILINE)

def _tool_docs() -> str:
    return "\n".join(f"- {name}: {fn.__doc__.strip()}" for name, fn in TOOLS.items())

AGENT_SYSTEM = f"""You are a helpful assistant that can use tools.

You have access to these tools:
{_tool_docs()}

To call a tool, reply in EXACTLY this format (nothing after it):
Thought: <your short reasoning>
Action: <tool_name>
Action Input: <a JSON object of the tool's arguments>

You will then receive an "Observation:" with the tool's result.
When you have the final answer, reply in this format instead:
Final: <your answer>

Rules:
- Use at most ONE tool call per message.
- Never put Action and Final in the same message.
- Example tool call:
Thought: I should compute this.
Action: calculator
Action Input: {{"expression": "3111696 / 74088"}}
"""

def _parse(model_text: str):
    """Return (tool_name, kwargs, final_text). Tool call takes priority over Final."""
    m = ACTION_RE.search(model_text)
    if m:
        name = m.group("name")
        after = model_text[m.end():]
        m_in = re.search(r"^Action Input:\s*(?P<rest>[\s\S]+)$", after, re.MULTILINE)
        raw = (m_in.group("rest").strip() if m_in else "").splitlines()
        raw = raw[0] if raw else "{}"
        try:
            kwargs = json.loads(raw)
            if not isinstance(kwargs, dict):
                kwargs = {"expression": str(kwargs)}
        except json.JSONDecodeError:
            kwargs = {"expression": raw}   # tolerate a bare expression
        return name, kwargs, None
    m = FINAL_RE.search(model_text)
    if m:
        return None, None, m.group("final").strip()
    return None, None, None

def run_agent(user_prompt: str, *, max_steps: int = 8, verbose: bool = True) -> str:
    history: list[dict] = [{"role": "user", "content": [{"type": "text", "text": user_prompt}]}]
    for step in range(1, max_steps + 1):
        text = qwen_chat(AGENT_SYSTEM, "", history=history)
        if verbose:
            print(f"\n--- step {step} ---\n{text}")
        name, kwargs, final = _parse(text)
        history.append({"role": "assistant", "content": [{"type": "text", "text": text}]})

        if final is not None:
            return final
        if name is None:            # no tool, no Final -> treat as the answer
            return text
        if name not in TOOLS:
            obs = f"ERROR: unknown tool {name!r}. Available: {sorted(TOOLS)}"
        else:
            try:
                obs = str(TOOLS[name](**kwargs))
            except Exception as e:
                obs = f"ERROR running {name}: {type(e).__name__}: {e}"
        if verbose:
            print(f"[tool] {name}({kwargs}) -> {obs}")
        history.append({"role": "user", "content": [{"type": "text", "text": f"Observation: {obs}"}]})
    return "ERROR: exceeded max_steps"

## Part 3 - Run the agent

Now the same questions that stumped the plain LLM in Part 1 — this time the
agent can call `current_time`, `calculator`, and `letter_counter`. Watch the
Thought → Action → Observation trace.

In [ ]:
query = """
Answer the following questions:
1. What is the current time in UTC?
2. Tell me how many letter R's are in the word "strawberry"
"""
answer = run_agent(query)
print("\n=== FINAL ANSWER ===")
print(answer)

In [ ]:
message = """
I have 3 requests:

1) What is the time right now (UTC)?
2) Calculate 3111696 / 74088
3) Then summarize the results in one short sentence.
"""
answer = run_agent(message)
print("\n=== FINAL ANSWER ===")
print(answer)

In [ ]:
# A harder one: the agent can compute, but a single pass may still reason wrong.
# (The companion notebook `agent_reflexion.ipynb` adds a verify-and-retry loop for this.)
puzzle = "In 1988, a person's age was equal to the sum of the digits of their birth year. How old was this person?"
print(run_agent(puzzle))

## Part 4 - Now build your own agent!
For example:
- Your personalized AI Study Buddy 🙂
- Autonomous Research Agent

When designing your own application-oriented agent, consider:
- your goal (what should it accomplish?)
- the tools required — add functions to the `TOOLS` dict above
- the system prompt (`AGENT_SYSTEM`)
- the model and your deployment environment

Tip: to add a tool, write a plain Python function with a one-line docstring and
register it in `TOOLS` — the loop and prompt pick it up automatically.

In [30]:
### your First Agent ###